# MiniDance · 训练（Colab）

在 Colab 上训练视频 DiT。S4 用它来测速度、算成本：先训练 1 小时，看效果怎么变化。

- **请在 Colab 网页版中运行**（需要挂载 Google 网盘）
- 运行时：**A100**（L4 也能跑，把 `BATCH` 调小）
- 需要网盘里已有编码好的 latent：`MyDrive/minidance/latents/`（由 `encode_latents_colab.ipynb` 生成）
- 可选：把短字幕文件 `short_captions.jsonl` 上传到 `MyDrive/minidance/`；没有的话只用原始长字幕训练
- checkpoint 和样例 GIF 都存到网盘，断开后重新运行会自动续训

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!free -g | head -2

## 1. 挂载网盘，获取代码

In [ ]:
REPO_URL = 'https://github.com/JuniverseLi/minidance.git'
DRIVE_DIR = '/content/drive/MyDrive/minidance'

import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    raise RuntimeError('网盘挂载失败，请在 Colab 网页版中打开本 notebook 运行') from e

if not os.path.exists('/content/minidance'):
    !git clone -q {REPO_URL} /content/minidance
else:
    !git -C /content/minidance fetch -q origin
    !git -C /content/minidance reset -q --hard origin/main
%cd /content/minidance
!git log --oneline -1

In [ ]:
!pip install -q diffusers safetensors "transformers>=4.51"
import transformers; print('transformers', transformers.__version__)

## 2. 把数据从网盘复制到本地磁盘

直接从网盘读太慢。约 10GB，复制一次大约几分钟；已经复制过的文件会跳过。

In [ ]:
import glob, time
SRC, DST = f'{DRIVE_DIR}/latents', '/content/latents'
os.makedirs(DST, exist_ok=True)
print(len(glob.glob(f'{SRC}/shard_*.safetensors')), 'shards on Drive')
t = time.time()
!cp -n {SRC}/shard_* {DST}/
n = len(glob.glob(f'{DST}/shard_*.safetensors'))
size = sum(os.path.getsize(p) for p in glob.glob(f'{DST}/shard_*')) / 2**30
print(f'{n} shards, {size:.1f} GB local, copy took {time.time() - t:.0f}s')

CAPS = f'{DRIVE_DIR}/short_captions.jsonl'
if not os.path.exists(CAPS):
    print('未找到短字幕文件，只用长字幕训练')
    CAPS = 'none'

## 3. 训练

- `MINUTES`：训练多久后自动停止（会先生成样例、保存 checkpoint）。S4 先跑 60 分钟。
- `BATCH`：A100 40GB 用 64；L4 24GB 用 32；显存不够（CUDA out of memory）就减半。
- 每 1000 步生成一次样例 GIF，存到 `DRIVE_DIR/runs/<RUN>/samples/`。
- `--compile` 能提速，如果报错就把它删掉再运行。
- 中途断开：重新运行全部单元格，会从网盘里的 checkpoint 继续。

In [ ]:
RUN = 's4_base'
MINUTES = 60
BATCH = 64
OUT = f'{DRIVE_DIR}/runs/{RUN}'

!python trainer/train_pretrain.py --data_dir /content/latents --short_captions {CAPS}     --preset base --batch {BATCH} --lr 2e-4 --warmup 1000 --steps 200000     --max_minutes {MINUTES} --sample_every 1000 --save_every 2000 --log_every 50     --compile --resume --out {OUT}

## 4. 查看样例

In [ ]:
from IPython.display import Image, display
gifs = sorted(glob.glob(f'{OUT}/samples/*.gif'))
print(len(gifs), 'sample GIFs; 每个 GIF 从左到右：4 条固定提示词 + 2 条训练集字幕')
for g in gifs[-3:]:
    print(os.path.basename(g))
    display(Image(filename=g))